In [10]:
kanda_number = 2
file_path = r"gpt_out_full_ramayan.json"
#read and parse that json file
import json
with open(file_path, 'r') as f:
    data = json.load(f)
kanda = []
for i in data:
    if i['Part'] == str(kanda_number):
        kanda.append(i)

with open(f'ayodhyakanda.json', 'w') as f:
    json.dump(kanda, f, indent=4)


In [12]:
file = 'ayodhyakanda.json'
import json
with open(file) as f:
    data = json.load(f)

print(len(data))

423


In [13]:
for i in data[:]:
    from token_count import TokenCount
    tc = TokenCount(model_name="gpt-3.5-turbo")
    text = i['Prompt_Answer']
    tokens = tc.num_tokens_from_string(text)
    print(f"Tokens in the string: {tokens}, {len(i['Prompt_Answer'])}")

Tokens in the string: 243, 1058
Tokens in the string: 381, 1945
Tokens in the string: 309, 1595
Tokens in the string: 283, 1263
Tokens in the string: 96, 439
Tokens in the string: 131, 607
Tokens in the string: 223, 1129
Tokens in the string: 288, 1366
Tokens in the string: 204, 1015
Tokens in the string: 175, 811
Tokens in the string: 168, 772
Tokens in the string: 265, 1107
Tokens in the string: 199, 882
Tokens in the string: 284, 1258
Tokens in the string: 116, 537
Tokens in the string: 157, 687
Tokens in the string: 215, 1066
Tokens in the string: 217, 1014
Tokens in the string: 163, 750
Tokens in the string: 188, 807
Tokens in the string: 245, 1103
Tokens in the string: 91, 427
Tokens in the string: 216, 995
Tokens in the string: 226, 1049
Tokens in the string: 153, 694
Tokens in the string: 318, 1413
Tokens in the string: 194, 837
Tokens in the string: 218, 981
Tokens in the string: 73, 310
Tokens in the string: 269, 1151
Tokens in the string: 271, 1215
Tokens in the string: 246,

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import StorageContext, load_index_from_storage
from custom_llm import OurLLM
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import KnowledgeGraphIndex, SimpleDirectoryReader
from llama_index.core import StorageContext

In [ ]:
llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))

In [ ]:
# from llama_index.core import Document
# from llama_index.core.schema import MetadataMode

# documents = []

# for i in data[:]:       
#     document = Document(
#     text = i['Prompt_Answer'],
#     metadata={
#         "canto_summary": i['Summary'],
#         "r_id": str(i['Canto']) + "_" + str(i['Prompt_Number'])
#     },
#     excluded_llm_metadata_keys=["r_id"],
#     metadata_seperator="::",
#     metadata_template="{value}",
#     text_template="Content: {content}, Chapter context: {metadata_str}\n",
#     )
#     documents.append(document)

In [ ]:
# index = VectorStoreIndex.from_documents(documents)

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage
# index.storage_context.persist(persist_dir="balakanda")
storage_context = StorageContext.from_defaults(persist_dir="balakanda")
index = load_index_from_storage(storage_context)

In [ ]:
base_retriever = index.as_retriever(similarity_top_k=5)
from llama_index.core import PromptTemplate

qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)


class RAGStringQueryEngine(CustomQueryEngine):
    """RAG String Query Engine."""

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: OurLLM
    qa_prompt: PromptTemplate

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)

        context_str = "\n\n".join([n.node.get_content() for n in nodes])
        response = self.llm.complete(
            qa_prompt.format(context_str=context_str, query_str=query_str)
        )

        return str(response)

query_engine = RAGStringQueryEngine(
    retriever=base_retriever,
    response_synthesizer=get_response_synthesizer(),
    llm=llmR,
    qa_prompt=qa_prompt,
)

In [ ]:
response = query_engine.query("""Who is Rama""") 

In [ ]:
from llama_index.core import KnowledgeGraphIndex, SimpleDirectoryReader
from llama_index.core import StorageContext
from llama_index.graph_stores.neo4j import Neo4jGraphStore